In [1]:
# Import libraries
import pandas as pd

In [2]:
# Load the collision data
# Use the sample dataset present in the script and save it in the same directory as the notebook
collisions = pd.read_csv("sample_collisions.csv")
print("Collisions data loaded:", collisions.shape)

# Load the parties (drivers/passengers) data
parties = pd.read_csv("sample_parties.csv")
print("Parties data loaded:", parties.shape)

Collisions data loaded: (935791, 65)
Parties data loaded: (1866917, 24)


In [3]:
# Real-world note:
# In real data projects, the transformation stage often takes 70–80% of the total effort.
# This includes cleaning column names, fixing formats, merging files, and filling missing values.
# These steps are essential before any meaningful analysis can be done.

In [4]:
# Step 1: Clean column names
collisions.columns = collisions.columns.str.lower().str.strip().str.replace(" ", "_")
parties.columns = parties.columns.str.lower().str.strip().str.replace(" ", "_")
print("Step 1: Cleaned column names")
display(collisions.head(2))
display(parties.head(2))

Step 1: Cleaned column names


,case_id,jurisdiction,officer_id,reporting_district,chp_shift,population,county_city_location,county_location,special_condition,beat_type,...,pedestrian_injured_count,bicyclist_killed_count,bicyclist_injured_count,motorcyclist_killed_count,motorcyclist_injured_count,latitude,longitude,collision_date,collision_time,process_date
0,8000993.0,1942.0,39335,1676,not chp,>250000,1942,los angeles,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2016-03-20,16:30:00,2016-03-30
1,4232117.0,3008.0,1284,Z2,not chp,100000 to 250000,3008,orange,0.0,not chp,...,0,0,0,0,0.0,NaN,NaN,2009-04-13,18:24:00,2009-11-30


,id,case_id,party_number,party_type,at_fault,party_sex,party_age,party_sobriety,direction_of_travel,party_safety_equipment_1,...,other_associate_factor_1,party_number_killed,party_number_injured,movement_preceding_collision,vehicle_year,vehicle_make,statewide_vehicle_type,chp_vehicle_type_towing,chp_vehicle_type_towed,party_race
0,1292305,4843348.0,1,driver,1,male,28.0,had not been drinking,south,lap/shoulder harness used,...,inattention,0,0,making right turn,1974.0,chevrolet,passenger car,NaN,NaN,NaN
1,3252679,5965300.0,2,driver,0,male,53.0,had not been drinking,south,air bag not deployed,...,entering/leaving ramp,0,0,proceeding straight,2000.0,toyota,passenger car,"passenger car, station",00,hispanic


In [5]:
# Step 2: Select only required columns
collisions_subset = collisions[["case_id", "county_location", "collision_date"]]
parties_subset = parties[["case_id", "party_age", "party_sobriety"]]
print("Step 2: Selected relevant columns")
display(collisions_subset.head(2))
display(parties_subset.head(2))

Step 2: Selected relevant columns


,case_id,county_location,collision_date
0,8000993.0,los angeles,2016-03-20
1,4232117.0,orange,2009-04-13


,case_id,party_age,party_sobriety
0,4843348.0,28.0,had not been drinking
1,5965300.0,53.0,had not been drinking


In [6]:
# Step 3: Rename columns to be more readable
collisions_subset = collisions_subset.rename(
    columns={"county_location": "county", "collision_date": "date"}
)
parties_subset = parties_subset.rename(
    columns={"party_age": "age", "party_sobriety": "sobriety"}
)
print("Step 3: Renamed columns")
display(collisions_subset.head(2))
display(parties_subset.head(2))

Step 3: Renamed columns


,case_id,county,date
0,8000993.0,los angeles,2016-03-20
1,4232117.0,orange,2009-04-13


,case_id,age,sobriety
0,4843348.0,28.0,had not been drinking
1,5965300.0,53.0,had not been drinking


In [7]:
# Step 4: Convert data types
collisions_subset["date"] = pd.to_datetime(collisions_subset["date"], errors="coerce")
parties_subset["age"] = pd.to_numeric(parties_subset["age"], errors="coerce")
print("Step 4: Converted types")
display(parties_subset[["age"]].head(2))
display(collisions_subset[["date"]].head(2))

Step 4: Converted types


,age
0,28.0
1,53.0


,date
0,2016-03-20
1,2009-04-13


In [8]:
# Step 5: Fill missing values
parties_subset["age"] = parties_subset["age"].fillna(-1)
parties_subset["sobriety"] = parties_subset["sobriety"].fillna("Unknown")
print("Step 5: Filled missing values")
display(parties_subset.head(2))

Step 5: Filled missing values


,case_id,age,sobriety
0,4843348.0,28.0,had not been drinking
1,5965300.0,53.0,had not been drinking


In [9]:
# Step 6: Standardize text
collisions_subset["county"] = collisions_subset["county"].str.title().str.strip()
parties_subset["sobriety"] = parties_subset["sobriety"].str.title().str.strip()
print("Step 6: Standardized text fields")
display(collisions_subset.head(2))
display(parties_subset.head(2))

Step 6: Standardized text fields


,case_id,county,date
0,8000993.0,Los Angeles,2016-03-20
1,4232117.0,Orange,2009-04-13


,case_id,age,sobriety
0,4843348.0,28.0,Had Not Been Drinking
1,5965300.0,53.0,Had Not Been Drinking


In [10]:
# Step 7: Filter out invalid ages
parties_subset = parties_subset[parties_subset["age"] >= 0]
print("Step 7: Filtered invalid age records")
display(parties_subset.head(2))

Step 7: Filtered invalid age records


,case_id,age,sobriety
0,4843348.0,28.0,Had Not Been Drinking
1,5965300.0,53.0,Had Not Been Drinking


In [11]:
# Step 8: Create a derived column — age group
parties_subset["age_group"] = pd.cut(
    parties_subset["age"],
    bins=[0, 18, 30, 45, 60, 120],
    labels=["<18", "18-30", "31-45", "46-60", "60+"],
)
print("Step 8: Created age groups")
display(parties_subset[["age", "age_group"]].head(2))

Step 8: Created age groups


,age,age_group
0,28.0,18-30
1,53.0,46-60


In [12]:
# Step 9: Drop duplicate rows (if any)
before = parties_subset.shape[0]
parties_subset = parties_subset.drop_duplicates()
after = parties_subset.shape[0]
print("Step 9: Dropped duplicates ({before - after} removed)")

Step 9: Dropped duplicates ({before - after} removed)


In [13]:
# Step 10: Merge datasets on case_id
merged_df = pd.merge(collisions_subset, parties_subset, on="case_id", how="inner")
print("Step 10: Merged datasets → shape:", merged_df.shape)
display(merged_df.head(2))

Step 10: Merged datasets → shape: (156323, 6)


,case_id,county,date,age,sobriety,age_group
0,1665179.0,Los Angeles,2004-09-18,48.0,Not Applicable,46-60
1,90608430.0,Los Angeles,2017-11-19,38.0,Had Not Been Drinking,31-45


In [14]:
# Step 11: Sort the data by date and age
merged_df = merged_df.sort_values(by=["date", "age"], ascending=[False, True])
print("Step 11: Sorted by date and age")
display(merged_df.head(2))

Step 11: Sorted by date and age


,case_id,county,date,age,sobriety,age_group
98507,91489025.0,Santa Clara,2021-06-02,53.0,Had Not Been Drinking,46-60
57509,81488075.0,Orange,2021-05-31,34.0,Had Not Been Drinking,31-45


In [15]:
# In a real production pipeline, this data would likely go into:
# - a relational database (e.g. PostgreSQL, MySQL)
# - a data lake (e.g. AWS S3, Azure Data Lake)
# - or into a dashboard tool like Power BI, Tableau, or Superset.

# For this project, we'll save it as a clean CSV.
merged_df.to_csv("crash_report_pandas.csv", index=False)
print("Final cleaned data saved to 'crash_report_pandas.csv'")

Final cleaned data saved to 'crash_report_pandas.csv'
